In [ ]:
# 1セルで Gemma 2 9B をインストールしてチャットする
# GPU があれば 4bit 量子化を使う。GPU がなければ CPU フォールバック。
# 使い方: このセルを実行して、"You:" と入力して会話。終了は exit / quit

%pip install -q --upgrade pip
%pip install -q "transformers>=4.45.0" "accelerate>=0.34.0" "torch>=2.3.0" "sentencepiece" "protobuf" "huggingface_hub"

# GPU のときだけ 4bit 量子化用のライブラリを入れる
try:
    import torch
    if torch.cuda.is_available():
        !pip install -q bitsandbytes
except Exception:
    pass

import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

MODEL_ID = 'google/gemma-2-9b-it'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

# 量子化設定（GPU のときのみ）
quantization_config = None
if device == 'cuda':
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map='auto',
    torch_dtype=torch.bfloat16 if device == 'cuda' else torch.float32,
)

messages = []
print('Gemma ready. Type your message. Enter exit or quit to end.')
while True:
    user_input = input('You: ').strip()
    if user_input.lower() in {'exit', 'quit'}:
        print('Goodbye!')
        break

    messages.append({'role': 'user', 'content': user_input})
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors='pt'
    ).to(model.device)

    with torch.inference_mode():
        outputs = model.generate(
            input_ids=inputs,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
        )

    generated = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True).strip()
    print(f'Gemma: {generated}')
    messages.append({'role': 'assistant', 'content': generated})
